# Recording dataset exploratory analysis

Select a dataset in the configuration cell and run all cells. The notebook reads the selected
folder's train/validation/test manifests and waveform clips, reports class and recording-group
counts, examines audio quality, and displays the exact log-mel model inputs. It supports
`training_dataset_v2`, `training_dataset_v3`, `training_dataset_v4`, or another dataset following
`DATA_CONTRACT.md`. All plots and findings are derived from the selected dataset.

Plots and audio playback have separate cells. Results are saved separately for each dataset.
Per-clip spectrogram image export is optional and uses the reusable `iml.export_spectrograms`
module. This notebook does not train a model, modify data, or search for original recordings.

Train, validation, and test are inspected descriptively. These test plots should not become
another model-selection objective. Recording IDs are grouping proxies, not verified independent
sessions; speaker/source metadata describe associations rather than causes of model behavior.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import torch
from IPython.display import Audio, Image, Markdown, display

from iml.config import ExperimentConfig
from iml.contract import CLIP_SAMPLES, SAMPLE_RATE, fingerprint, validate_dataset
from iml.data import LogMelSpectrogram, SpectrogramDataset, crop_or_pad, load_clip, normalize
from iml.export_spectrograms import export_spectrograms, draw_spectrogram
from iml.preprocess import frame_db

# Change this setting to training_dataset_v3 or training_dataset_v4, then run all cells.
# IML_EDA_DATASET can override it for reproducible batch execution.
DATASET_NAME = os.environ.get("IML_EDA_DATASET", "training_dataset_v2")
EXPORT_ALL_SPECTROGRAMS = False  # set True to write one PNG per clip; sample plots always run
SEED = 42

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "DATA_CONTRACT.md").exists())
DATA = (ROOT / "data" / DATASET_NAME).resolve()
PICTURES = DATA / "spectrograms"
REPORTS = ROOT / "notebooks/eda/reports" / DATA.name
REPORTS.mkdir(parents=True, exist_ok=True)
CFG = ExperimentConfig(model="efficientnetv2_s", sample_rate=16000, excerpt_sec=3.0,
                       n_fft=512, hop_length=160, n_mels=80, normalize_input="per_sample")
RAW_CFG = CFG.with_overrides("normalize_input=none")
FRONTEND = {k: getattr(CFG, k) for k in
            ("sample_rate", "excerpt_sec", "n_fft", "hop_length", "n_mels", "normalize_input")}
STORED_CLIP_SECONDS = CLIP_SAMPLES / SAMPLE_RATE
EXPECTED_FRAMES = int(CFG.sample_rate * CFG.excerpt_sec) // CFG.hop_length + 1
SPLITS = ["train", "val", "test"]
torch.set_num_threads(4)
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.facecolor": "white"})
errors = validate_dataset(DATA)
if errors:
    raise ValueError(errors)
BEFORE = fingerprint(DATA)
print("Python:", sys.executable)
print("Dataset:", DATA)
print("Frontend:", FRONTEND)
print("Fingerprint:", BEFORE["sha1"])
print("Reports:", REPORTS)

def save_figure(fig, name):
    fig.savefig(REPORTS / name, dpi=150, bbox_inches="tight")
    display(Image(filename=str(REPORTS / name)))
    plt.close(fig)

## Dataset composition and recording groups

The CSV `allowed` column determines the class. Optional `clip_provenance.csv` can supply
verified speaker/recording identifiers if it exactly matches the manifest paths and labels.
Otherwise the notebook uses the documented filename conventions: the final number is a clip
offset, local prefixes identify named people, N identifies external speakers, and VoxCeleb
prefixes identify its speakers. Unknown naming is grouped as Other, not assigned a class.

N and VoxCeleb groups contain multiple people. Filename-derived speaker/recording identifiers
are provisional; they cannot establish independence across sessions or identify unknown voices.

In [ ]:
frames = []
for split in SPLITS:
    part = pd.read_csv(DATA / f"{split}.csv")
    part["split"] = split
    frames.append(part)
clips = pd.concat(frames, ignore_index=True)
if clips.empty:
    raise ValueError("No clips are listed in the selected dataset")
assert clips.path.is_unique

def filename_identifiers(path):
    stem = Path(path).stem
    recording, separator, offset = stem.rpartition("_")
    recording = recording if separator and offset.isdigit() else stem
    tokens = recording.split("_")
    speaker = "_".join(tokens[:2]) if tokens[0] in {"vox1", "vox2", "tv"} else tokens[0]
    return recording, speaker

inferred = clips.path.map(filename_identifiers)
clips["recording_id"] = inferred.map(lambda pair: pair[0])
clips["speaker_id"] = inferred.map(lambda pair: pair[1])
clips["start_ms"] = pd.to_numeric(clips.path.str.extract(r"_(\d+)\.npy$", expand=False), errors="coerce")
identity_source = "documented filename conventions"
provenance_path = DATA / "clip_provenance.csv"
if provenance_path.is_file():
    provenance = pd.read_csv(provenance_path)
    required = {"path", "allowed", "speaker", "recording_id"}
    valid = required <= set(provenance.columns) and provenance.path.is_unique
    if valid:
        valid = set(provenance.path) == set(clips.path)
    if valid:
        by_path = provenance.set_index("path")
        valid = np.array_equal(clips.path.map(by_path.allowed).to_numpy(), clips.allowed.to_numpy())
    if valid:
        for column, source_column in (("speaker_id", "speaker"), ("recording_id", "recording_id")):
            mapped = clips.path.map(by_path[source_column])
            clips[column] = mapped.where(mapped.notna(), clips[column]).astype(str)
        identity_source = "clip_provenance.csv, verified against manifest paths and labels"
    else:
        print("Provenance does not exactly match the manifests; using filename identifiers.")

def display_group(speaker):
    if speaker in {"M", "P", "MS", "D", "T"}:
        return speaker
    if speaker.startswith("N") and speaker[1:].isdigit():
        return "External N"
    for prefix, name in (("vox1_", "VoxCeleb1"), ("vox2_", "VoxCeleb2"), ("tv_", "TidyVoice")):
        if speaker.startswith(prefix):
            return name
    return "Other"

clips["source_group"] = clips.speaker_id.map(display_group)
preferred = ["M", "P", "MS", "D", "T", "External N", "VoxCeleb1", "VoxCeleb2", "TidyVoice", "Other"]
GROUPS = [group for group in preferred if group in set(clips.source_group)]
COLORS = {group: plt.get_cmap("tab10")(i % 10) for i, group in enumerate(GROUPS)}
counts = pd.crosstab(clips.split, clips.allowed).reindex(index=SPLITS, columns=[0, 1], fill_value=0)
counts = counts.rename(columns={0: "invalid", 1: "allowed"})
counts["total"] = counts.sum(axis=1)
counts["invalid_per_allowed"] = counts.invalid / counts.allowed.replace(0, np.nan)
counts["unique_recordings"] = clips.groupby("split").recording_id.nunique().reindex(SPLITS, fill_value=0)
counts["summed_clip_minutes"] = counts.total * STORED_CLIP_SECONDS / 60
source_counts = pd.crosstab(clips.source_group, clips.split).reindex(index=GROUPS, columns=SPLITS, fill_value=0)
display(counts.round(3))
display(source_counts)
print(f"Total: {len(clips)} clips, {clips.recording_id.nunique()} recording IDs, "
      f"{clips.speaker_id.nunique()} inferred/metadata speaker IDs.")
print("Identifier source:", identity_source)
print("Summed clip duration includes overlapping audio; it is not independent source duration.")
counts.to_csv(REPORTS / "split_summary.csv")
source_counts.to_csv(REPORTS / "speaker_source_counts.csv")

## Waveform quality and exact model features

RMS/peak are measured relative to PCM full scale (dBFS). `low_energy_fraction` uses a fixed
−45 dBFS threshold on 20 ms frames as a descriptive measurement. It is **not** the producer's
speech ratio: the production VAD threshold depends on the full original recording, which
cannot be reconstructed from a stored clip. PCM-limit hits are counted exactly at −32768/32767.

The normalized tensor comes directly from `SpectrogramDataset`, using the same frontend as
training. Raw log-mel power is additionally shown to make the effect of per-sample normalization
visible. With native three-second inputs, evaluation and training cropping select the same audio.

In [ ]:
metrics = []
raw_specs = np.empty((len(clips), CFG.n_mels, EXPECTED_FRAMES), dtype=np.float32)
norm_specs = np.empty_like(raw_specs)
raw_frontend = LogMelSpectrogram(RAW_CFG)
with torch.inference_mode():
    for split in SPLITS:
        part = clips[clips.split.eq(split)]
        ds = SpectrogramDataset([(DATA / r.path, int(r.allowed)) for r in part.itertuples()],
                                CFG, train=False)
        for local_i, row in enumerate(part.itertuples()):
            pcm = np.load(DATA / row.path, allow_pickle=False)
            wave = crop_or_pad(load_clip(DATA / row.path, CFG.sample_rate), ds.n_samples)
            model_input, label = ds[local_i]
            raw = raw_frontend(wave)
            torch.testing.assert_close(model_input, normalize(raw, CFG.normalize_input).squeeze(0))
            assert label == row.allowed and tuple(model_input.shape) == (1, CFG.n_mels, EXPECTED_FRAMES)
            raw_specs[row.Index] = raw[0, 0].numpy()
            norm_specs[row.Index] = model_input[0].numpy()
            values = pcm.astype(np.float64) / 32768
            rms = float(np.sqrt(np.mean(values**2)))
            peak = float(np.max(np.abs(values)))
            metrics.append({"path": row.path, "spec_index": row.Index,
                "rms_dbfs": 20 * np.log10(max(rms, 1e-12)),
                "peak_dbfs": 20 * np.log10(max(peak, 1e-12)),
                "dc_offset": float(values.mean()),
                "low_energy_fraction": float(np.mean(frame_db(pcm) < -45)),
                "pcm_limit_fraction": float(np.mean((pcm == -32768) | (pcm == 32767))),
                "waveform_sha256": hashlib.sha256(pcm.tobytes()).hexdigest()})
features = clips.merge(pd.DataFrame(metrics), on="path", validate="one_to_one")
display(features.groupby("source_group")[
    ["rms_dbfs", "peak_dbfs", "low_energy_fraction", "pcm_limit_fraction"]
].median().reindex(GROUPS).round(4))
print("Verified every tensor against the shared frontend:", raw_specs.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), layout="constrained")
counts[["allowed", "invalid"]].plot.bar(ax=axes[0], color=["#2563eb", "#ea580c"], rot=0)
axes[0].set(title="Class balance by split", ylabel="Clips", xlabel="")
axes[0].legend(title="Class")
source_counts.plot.bar(stacked=True, ax=axes[1], color=["#2563eb", "#f59e0b", "#64748b"], rot=25)
axes[1].set(title="Clip distribution by speaker/source group", ylabel="Clips", xlabel="Speaker/source group")
axes[1].legend(title="Split")
save_figure(fig, "dataset_composition.png")

In [ ]:
recordings = features.groupby(["source_group", "recording_id", "split"], as_index=False).agg(
    clips=("path", "size"), rms_dbfs=("rms_dbfs", "median"))
recording_leaks = features.groupby("recording_id").split.nunique()
recording_leaks = recording_leaks[recording_leaks.gt(1)]
hash_groups = features.groupby("waveform_sha256")
exact_duplicate_groups = int(hash_groups.size().gt(1).sum())
cross_split_exact_duplicates = int(hash_groups.split.nunique().gt(1).sum())
overlap_rows = []
for (recording, split), group in clips.groupby(["recording_id", "split"]):
    starts = np.sort(group.start_ms.dropna().to_numpy())
    overlap_rows.append({"recording_id": recording, "split": split,
                         "adjacent_overlapping_pairs": int(np.sum(np.diff(starts) < STORED_CLIP_SECONDS * 1000))})
overlaps = pd.DataFrame(overlap_rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), layout="constrained")
features.groupby("source_group").recording_id.nunique().reindex(GROUPS).plot.bar(
    ax=axes[0], color=[COLORS[g] for g in GROUPS], rot=25)
axes[0].set(title="Recording IDs by speaker/source group", ylabel="Recording IDs", xlabel="")
for group in GROUPS:
    x = recordings.loc[recordings.source_group.eq(group), "clips"].sort_values().to_numpy()
    cumulative_percent = 100 * np.arange(1, len(x) + 1) / len(x)
    axes[1].step(np.r_[0, x, recordings.clips.max()], np.r_[0, cumulative_percent, 100],
                 where="post", label=group, color=COLORS[group])
axes[1].set(title="How many clips come from one recording?", xlabel="Clips per recording", ylabel="Recordings with this many clips or fewer (%)", ylim=(0, 102))
axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))
axes[1].legend(title="Group", ncol=2, fontsize=8)
save_figure(fig, "recording_groups.png")
print("Recording IDs crossing splits:", len(recording_leaks))
print("Exact waveform duplicate groups:", exact_duplicate_groups,
      "; duplicate hashes crossing splits:", cross_split_exact_duplicates)
if not recording_leaks.empty:
    display(recording_leaks.rename("split_count").to_frame())
display(overlaps.groupby("split").adjacent_overlapping_pairs.sum().reindex(SPLITS, fill_value=0))
print("Overlaps use known filename offsets; unknown offsets are excluded. This is not a near-duplicate or session audit.")
recordings.to_csv(REPORTS / "recording_summary.csv", index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), layout="constrained")
for ax, column, title, ylabel in [
    (axes[0, 0], "rms_dbfs", "Typical waveform level", "RMS (dBFS)"),
    (axes[0, 1], "peak_dbfs", "Waveform peaks", "Peak (dBFS)"),
    (axes[1, 0], "low_energy_fraction", "Low-energy 20 ms frames", "Fraction below −45 dBFS"),
]:
    arrays = [features.loc[features.source_group.eq(g), column].to_numpy() for g in GROUPS]
    bp = ax.boxplot(arrays, tick_labels=GROUPS, patch_artist=True,
                    flierprops={"markersize": 2, "alpha": .4})
    for box, group in zip(bp["boxes"], GROUPS):
        box.set_facecolor(COLORS[group]); box.set_alpha(.55)
    ax.set(title=title, ylabel=ylabel, xlabel="Speaker/source group")
    ax.tick_params(axis="x", labelrotation=25)
clipped = features.assign(has_limit=features.pcm_limit_fraction.gt(0)).groupby("source_group").has_limit.mean()
axes[1, 1].bar(GROUPS, 100 * clipped.reindex(GROUPS), color=[COLORS[g] for g in GROUPS])
axes[1, 1].set(title="Clips containing PCM-limit samples", ylabel="Clips (%)", xlabel="Speaker/source group", ylim=(0, 100))
axes[1, 1].tick_params(axis="x", labelrotation=25)
save_figure(fig, "waveform_quality.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
for group in GROUPS:
    indices = features.loc[features.source_group.eq(group), "spec_index"].to_numpy()
    for ax, values in zip(axes, (raw_specs, norm_specs)):
        curves = values[indices].mean(axis=2)
        lo, med, hi = np.quantile(curves, [.25, .5, .75], axis=0)
        ax.plot(np.arange(CFG.n_mels), med, label=group, color=COLORS[group])
        ax.fill_between(np.arange(CFG.n_mels), lo, hi, color=COLORS[group], alpha=.09)
axes[0].set(title="Log-mel features before input normalization", ylabel="Time-mean log-mel power (dB)", xlabel="Mel band")
axes[1].set(title="Log-mel features after input normalization", ylabel="Time-mean normalized log-mel", xlabel="Mel band")
axes[1].legend(title="Speaker/source group", ncol=3)
save_figure(fig, "spectral_profiles.png")
print("Curves are medians with interquartile bands over clips. Differences can reflect voice, recording channel, or both.")

## Matched waveform, raw log-mel, and actual model input

Choose the clip closest to the median RMS within each group, rather than choosing by a model
prediction or a visually convenient result. All raw log-mel panels use the same dB color range;
normalized panels use the exporter's shared range. Color saturation affects display only.

Both spectrogram columns use the same stored clip. This compares feature normalization, not original versus processed audio or an audio-loudness change.

In [ ]:
examples = []
for group in GROUPS:
    part = features[features.source_group.eq(group)]
    example_i = (part.rms_dbfs - part.rms_dbfs.median()).abs().idxmin()
    examples.append(features.loc[example_i])
examples = pd.DataFrame(examples)
raw_limits = (float(raw_specs.min()), float(raw_specs.max()))
fig, axes = plt.subplots(len(GROUPS), 3, figsize=(16, max(4, 2.5 * len(GROUPS))), layout="constrained", squeeze=False)
for row_i, row in enumerate(examples.itertuples()):
    wave = load_clip(DATA / row.path, CFG.sample_rate).numpy()
    axes[row_i, 0].plot(np.arange(len(wave)) / CFG.sample_rate, wave, color=COLORS[row.source_group], lw=.5)
    axes[row_i, 0].set(title=f"{row.source_group}: allowed={row.allowed}, {row.split}",
                       xlabel="Time (s)", ylabel="Amplitude / full scale", ylim=(-1, 1))
    raw_image = draw_spectrogram(axes[row_i, 1], raw_specs[row.spec_index], RAW_CFG, title="Log-mel before input normalization")
    raw_image.set_clim(*raw_limits)
    fig.colorbar(raw_image, ax=axes[row_i, 1], label="dB")
    normalized_image = draw_spectrogram(axes[row_i, 2], norm_specs[row.spec_index], CFG, title="Model input")
    fig.colorbar(normalized_image, ax=axes[row_i, 2], label="Standard deviations", extend="both")
save_figure(fig, "waveforms_and_model_inputs.png")
display(examples[["source_group", "path", "allowed", "split", "rms_dbfs"]].reset_index(drop=True))

In [ ]:
# Playback preserves relative digital amplitude; IPython Audio's automatic normalization is disabled.
for row in examples.itertuples():
    display(Markdown(f"**{row.source_group}**, allowed={row.allowed}, {row.split}: `{row.path}`"))
    display(Audio(load_clip(DATA / row.path, CFG.sample_rate).numpy(), rate=CFG.sample_rate, normalize=False))

## Optional: export a picture for every clip

Set `EXPORT_ALL_SPECTROGRAMS = True` in the configuration cell to export to the selected
dataset's `spectrograms/` folder. Otherwise the notebook produces only its analysis figures.
For a separate export, use the reusable script with the desired input and output folders:

```bash
python -m iml.export_spectrograms --data data/training_dataset_v3 --out /tmp/v3-spectrograms
```

The pictures render actual normalized model tensors, grouped by split/class with an index.
They are visualizations, not alternate training inputs. Unchanged exports can be resumed.

In [ ]:
image_index = None
if EXPORT_ALL_SPECTROGRAMS:
    image_index = export_spectrograms(DATA, PICTURES, CFG, verbose=True)
    assert len(image_index) == len(clips)
    assert all((PICTURES / p).is_file() for p in image_index.image)
    display(image_index.groupby(["split", "allowed"]).size().rename("exported PNGs").to_frame())
    print("Image folder:", PICTURES)
else:
    print("Full per-clip image export is disabled. Set EXPORT_ALL_SPECTROGRAMS=True to enable it.")

In [ ]:
# Findings and reproducibility metadata for the selected dataset, not a fixed snapshot.
AFTER = fingerprint(DATA)
assert AFTER == BEFORE
positive_recordings = features.loc[features.allowed.eq(1), "recording_id"].nunique()
positive_val_recordings = features.loc[features.allowed.eq(1) & features.split.eq("val"), "recording_id"].nunique()
group_rms = features.groupby("source_group").rms_dbfs.median()
quietest, loudest = group_rms.idxmin(), group_rms.idxmax()
export_note = (f"Exported/reused {len(image_index)} clip pictures under `{PICTURES}`."
               if image_index is not None else "Full per-clip image export was disabled; analysis figures were saved.")
findings = f"""
### Observations: {DATA.name}

- **{len(clips)} clips**, **{clips.recording_id.nunique()} recording IDs**, and **{clips.speaker_id.nunique()} inferred/metadata speaker IDs**. Identifiers use {identity_source}.
- Allowed-class clips cover **{positive_recordings} recording IDs**, including **{positive_val_recordings}** in validation. Correlated excerpts mean clip count overstates independent evidence.
- **{len(recording_leaks)} recording IDs cross splits**. There are **{exact_duplicate_groups} exact waveform duplicate groups**, with **{cross_split_exact_duplicates}** crossing splits. These checks do not establish absence of acoustic near duplicates or shared sessions.
- **{int(features.pcm_limit_fraction.gt(0).sum())} clips** contain int16 PCM-limit samples. This does not prove analog clipping or rule out other distortion.
- Median RMS spans **{group_rms.max() - group_rms.min():.1f} dB** across displayed groups ({quietest}: {group_rms.min():.1f} dBFS; {loudest}: {group_rms.max():.1f} dBFS). Voice and recording conditions may both contribute.
- All **{len(clips)} model inputs** match the shared frontend and have shape **{CFG.n_mels} × {EXPECTED_FRAMES}**. Normalization changes feature scale; it does not denoise waveforms.
- {export_note}
- Dataset fingerprint unchanged: `{AFTER['sha1']}`. No waveform, class label, or split was modified.
"""
display(Markdown(findings))
(REPORTS / "observations.md").write_text(findings.strip() + "\n")
metadata = {"dataset": AFTER, "frontend": FRONTEND, "seed": SEED,
            "identity_source": identity_source, "export_all_spectrograms": EXPORT_ALL_SPECTROGRAMS,
            "versions": {"python": sys.version.split()[0], "numpy": np.__version__,
                         "pandas": pd.__version__, "torch": str(torch.__version__),
                         "matplotlib": matplotlib.__version__}}
(REPORTS / "eda_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print("Figures, measurements, and reproducibility metadata:", REPORTS)